As an example of larger class, let's try implementing a new datatype `Fraction` to represent and calculate with rational numbers.

We need at least two member variables; the numerator and denominator:

In [11]:
class Fraction:
    def __init__(self, num, den):
        self.num = num
        self.den = den


frac = Fraction(2, 3)
frac.num, frac.den

(2, 3)

Nice, but this is still just container a container for data. It would be great if we could at least display the fraction properly. To do so, we need to add the method `__repr__` to the class definition. This dunder method is automatically called when Python tries to print the object:

In [13]:
class Fraction:
    def __init__(self, num, den):
        self.num = num
        self.den = den

    def __repr__(self):
        return f'{self.num}/{self.den}'


frac = Fraction(4, 5)
frac

4/5

That looks like a fraction.

Now let's add a function `set()` that allows the user to modify the numerator and denominator. We can then clean up the API by hiding the internal attributes from the user by prepending an underscore to `num` and `den`:

In [50]:
class Fraction:
    def __init__(self, num=0, den=1):
        self._num = num
        self._den = den

    def __repr__(self):
        return f'{self._num}/{self._den}'

    def set(self, num=None, den=None):
        if num is not None:
            self._num = num
        if den is not None:
            self._den = den


frac = Fraction(2, 3)
frac.set(num=42)
frac

42/3

With the definition in the cell above, we can initialize a `Fraction`, print it, and modify it. But, unfortunately, we can't use it in calculations yet because Python doesn't know what `+` or `-` means for a `Fraction`:

In [52]:
frac + frac

TypeError: unsupported operand type(s) for +: 'Fraction' and 'Fraction'

To make the `+` *operator* and other operators work, Python has many dunder methods that can be used to emulate numeric objects. For example:

* `__add__()` is called when `+` is used with the object on the left hand side;
* `__radd__()` is called when `+` is used with the object on the right hand side;
* `__sub__()` is called when `-` is used with the object on the left hand side.
* etc.

By implementing these methods, we can make our `Fraction` objects function as any other number.

The final `Fraction` class, which a polished API and supporting `+` and `-` could look like the implementation below.
Note how the helper function `_reduce()` has an underscore, and thus is hidden in the API.
Also, note how the `__sub__`, `__radd__` and `__rsub__` methods efficiently use the `__add__` and `__neg__` methods, preventing code duplication.

In [1]:
from math import gcd


class Fraction:
    """
    Represents a rational number as a numerator and denominator.
    Supports addition and subtraction with integer, floating point, and other rational numbers.
    Other mathematical operations, such as division and multiplication are not implemented.

    :param num: The numerator
    :param den: The denominator
    """
    def __init__(self, num=0, den=1):
        self._num = num
        self._den = den
        self._reduce()

    def __repr__(self):
        # Called when the fraction is displayed.
        if self._num == 0:
            return "0"
        if self._den == 1:
            return str(self._num)
        else:
            return f'{self._num}/{self._den}'

    def __neg__(self):
        # Called when '-fraction' is used.
        return Fraction(-self._num, self._den)

    def __add__(self, other):
        # Called when 'fraction + other' is used.
        if isinstance(other, Fraction):
            num = self._num * other._den + other._num * self._den
            den = self._den * other._den
            return Fraction(num, den)
        elif isinstance(other, int):
            num = self._num + other * self._den
            return Fraction(num, self._den)
        else:
            return self._num / self._den + other

    def __sub__(self, other):
        # Called when 'fraction - other' is used.
        return self + (-other)  # f - x = f + (-x)

    def __radd__(self, other):
        # Called when 'other + fraction' is used.
        return self + other  # x + f = f + x

    def __rsub__(self, other):
        # Called when 'other - fraction' is used.
        return -self + other  # x - f = -f + x

    def _reduce(self):
        # Helper function for simplifying the fraction. Converts e.g. 6/-4 to -3/2.

        # Find the greatest common divisor, and divide both the numerator and denominator by it.
        divisor = gcd(self._num, self._den)
        self._num //= divisor
        self._den //= divisor

        # Swap signs of fractions with negative denominators.
        if self._den < 0:
            self._num *= -1
            self._den *= -1

    def set(self, num=None, den=None):
        """
        Sets the numerator and/or denominator of the fraction.

        :param num: The numerator
        :param den: The denominator
        """
        if num is not None:
            self._num = num
        if den is not None:
            self._den = den
        self._reduce()

Let's do some tests:

In [12]:
a = Fraction(2, 4)
b = Fraction(5, 6)
c = 3
d = 3.14

print(f'{a} + {b} = {a + b}')
print(f'{a} - {b} = {a - b}')
print(f'{a} - {a} = {a - a}')
print(f'{a} + {c} = {a + c}')
print(f'{c} - {a} = {c - a}')
print(f'{d} - {a} = {d - a}')

1/2 + 5/6 = 4/3
1/2 - 5/6 = -1/3
1/2 - 1/2 = 0
1/2 + 3 = 7/2
3 - 1/2 = 5/2
3.14 - 1/2 = 2.64


## Try It Out Yourself

> **External course material**
>
> Review these pages and do the exercises:
>
> * <https://www.w3schools.com/python/python_challenges_class_methods.asp>
> * <https://www.w3schools.com/python/python_magic_methods.asp>
> * <https://www.w3schools.com/python/python_magic_eq.asp>
> * <https://www.w3schools.com/python/python_magic_lt.asp>